<a href="https://colab.research.google.com/github/MPMauricio/Calendarizacion-Ujieres-Antigravity2026/blob/main/pdvfinaljunio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# 📍 PDV FINDER PRO - VERSIÓN FINAL
# ============================================
!pip install pandas openpyxl -q

import pandas as pd
from datetime import datetime
import math
import json
import os
import zipfile

# ───────────────────────────────────────────
# FUNCIONES AUXILIARES
# ────────────────────────────────────────────
def formatear_dolares(valor):
    try:
        if pd.isna(valor) or str(valor) in ['ERROR:#N/A', 'N/A', '', '0']:
            return "$0.00"
        limpio = str(valor).replace('$', '').replace(',', '').strip()
        return f"${float(limpio):,.2f}"
    except:
        return "$0.00"

def limpiar_texto(valor):
    if pd.isna(valor) or str(valor) in ['ERROR:#N/A', 'N/A', '', 'nan']:
        return 'N/A'
    return str(valor).strip()

# ────────────────────────────────────────────
# 1. CARGA DE DATOS
# ────────────────────────────────────────────
print("="*70)
print(" PDV FINDER PRO - VERSION FINAL")
print("="*70)

from google.colab import files
print("\n Selecciona tu archivo Excel:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df_temp = pd.read_excel(filename)
df_temp = df_temp.dropna(how='all')

mapeo_cols = {
    'Ruta': 'Ruta',
    'Departamento': 'Departamento',
    'Frecuencia de visita': 'Frecuencia de visita',
    'Dias de visita': 'Dias de visita',
    'Categoria': 'Categoria',
    'Codigo de PDV': 'Codigo de PDV',
    'Numero de Recarga': 'Numero de  Recarga',
    'Nombre del PDV': 'Nombre del PDV',
    'Venta Promedio': 'Venta Promedio',
    'Latitud': 'Latitud',
    'Longitud': 'Longitud'
}

cols_existentes = df_temp.columns.tolist()
df_limpio = pd.DataFrame()

for nuevo, original in mapeo_cols.items():
    col_encontrada = None
    for col in cols_existentes:
        if col.strip().lower().replace('  ', ' ') == original.strip().lower().replace('  ', ' '):
            col_encontrada = col
            break
    if col_encontrada:
        df_limpio[nuevo] = df_temp[col_encontrada]
    else:
        for col in cols_existentes:
            if 'Numero' in col and 'Recarga' in col:
                col_encontrada = col
                df_limpio[nuevo] = df_temp[col_encontrada]
                break
        if not col_encontrada:
            df_limpio[nuevo] = 'N/A'

df_limpio['Latitud'] = pd.to_numeric(df_limpio['Latitud'], errors='coerce')
df_limpio['Longitud'] = pd.to_numeric(df_limpio['Longitud'], errors='coerce')
df_limpio = df_limpio.dropna(subset=['Latitud', 'Longitud'])

df_limpio['Venta Promedio Formato'] = df_limpio['Venta Promedio'].apply(formatear_dolares)
df_limpio['Venta Promedio Num'] = pd.to_numeric(
    df_limpio['Venta Promedio'].astype(str).str.replace(r'[^\d.]', '', regex=True),
    errors='coerce'
).fillna(0)

total_pdv = len(df_limpio)
print(f"\n {total_pdv} PDV cargados correctamente")

# ────────────────────────────────────────────
# 2. PREPARAR DATOS
# ────────────────────────────────────────────
pdv_data = []
for _, row in df_limpio.iterrows():
    pdv_data.append({
        'ruta': limpiar_texto(row['Ruta']),
        'departamento': limpiar_texto(row['Departamento']),
        'dias': limpiar_texto(row['Dias de visita']),
        'categoria': limpiar_texto(row['Categoria']),
        'codigo_pdv': limpiar_texto(row['Codigo de PDV']),
        'recarga': limpiar_texto(row['Numero de Recarga']),
        'nombre': limpiar_texto(row['Nombre del PDV']),
        'venta': row['Venta Promedio Formato'],
        'venta_num': float(row['Venta Promedio Num']),
        'lat': float(row['Latitud']),
        'lon': float(row['Longitud'])
    })

# ────────────────────────────────────────────
# 3. CONSTRUIR HTML - VERSION FINAL
# ────────────────────────────────────────────

html_content = '''<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no">
    <title>PDV Finder Pro</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&display=swap" rel="stylesheet">
    <style>
        :root { --primary: #4361ee; --secondary: #3f37c9; --accent: #4cc9f0; --danger: #f72585; --success: #06d6a0; --waze: #33ccff; --gmaps: #4285f4; --bg: #f8f9fa; --card-bg: #ffffff; --text: #2b2d42; }
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body { font-family: 'Poppins', sans-serif; background: var(--bg); color: var(--text); padding-bottom: 120px; -webkit-tap-highlight-color: transparent; }

        /* LOGIN */
        #loginOverlay { position: fixed; top: 0; left: 0; right: 0; bottom: 0; background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); z-index: 9999; display: flex; align-items: center; justify-content: center; padding: 20px; }
        .login-box { background: white; border-radius: 20px; padding: 30px; width: 100%; max-width: 400px; box-shadow: 0 20px 60px rgba(0,0,0,0.3); text-align: center; }
        .login-title { font-size: 22px; font-weight: 700; color: var(--text); margin: 10px 0 5px; }
        .login-subtitle { color: #666; font-size: 14px; margin-bottom: 25px; }
        .login-input { width: 100%; padding: 14px 16px; margin-bottom: 12px; border: 2px solid #e0e0e0; border-radius: 12px; font-size: 15px; font-family: 'Poppins', sans-serif; }
        .login-input:focus { outline: none; border-color: var(--primary); }
        .login-btn { width: 100%; padding: 14px; background: linear-gradient(135deg, #4361ee 0%, #3f37c9 100%); color: white; border: none; border-radius: 12px; font-size: 16px; font-weight: 600; cursor: pointer; margin-top: 10px; }
        .login-error { background: #ffebee; color: #c62828; padding: 10px 15px; border-radius: 8px; margin-bottom: 15px; font-size: 13px; display: none; }
        .login-footer { margin-top: 20px; font-size: 12px; color: #999; }

        /* HEADER */
        #appContent { display: none; }
        .header {
            background: linear-gradient(135deg, #3f37c9 0%, #4361ee 100%);
            padding: 15px 20px 20px;
            border-radius: 0 0 20px 20px;
            text-align: center;
            color: white;
            box-shadow: 0 4px 15px rgba(67, 97, 238, 0.3);
            position: relative;
            margin-bottom: 10px;
        }
        .header h1 { font-size: 20px; font-weight: 700; margin-bottom: 3px; }
        .header p { font-size: 13px; opacity: 0.9; }
        .btn-logout { position: absolute; right: 10px; top: 10px; background: rgba(255,255,255,0.2); color: white; border: none; padding: 6px 12px; border-radius: 15px; font-size: 11px; cursor: pointer; }

        /* BOTONES */
        .btn-main {
            background: linear-gradient(135deg, #f72585 0%, #ff6b6b 100%);
            color: white; border: none; padding: 14px 20px; border-radius: 12px;
            font-size: 15px; font-weight: 600; width: 90%; max-width: 320px;
            margin: -15px auto 15px; display: flex; align-items: center;
            justify-content: center; gap: 8px; box-shadow: 0 6px 15px rgba(247, 37, 133, 0.3);
            cursor: pointer;
        }
        .btn-secondary {
            background: linear-gradient(135deg, #06d6a0 0%, #118ab2 100%);
            color: white; border: none; padding: 10px 16px; border-radius: 10px;
            font-size: 13px; font-weight: 600; cursor: pointer; margin: 5px;
            display: inline-flex; align-items: center; gap: 6px;
        }
        .btn-secondary:active { opacity: 0.9; transform: scale(0.98); }

        /* CONTADOR Y BUSCADOR */
        .contador { text-align: center; padding: 8px 15px; color: #666; font-size: 12px; font-weight: 500; background: white; margin: 0 10px 10px; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.05); display: none; }
        .contador strong { color: var(--primary); }
        .modo-busqueda { background: #fff3cd; color: #856404; padding: 6px 12px; margin: 0 10px 8px; border-radius: 8px; font-size: 12px; text-align: center; display: none; font-weight: 500; }
        .modo-busqueda.active { display: block; }
        .search-container { padding: 0 15px; margin-bottom: 12px; }
        .search-box { background: white; border-radius: 12px; padding: 10px 12px; display: flex; align-items: center; box-shadow: 0 2px 8px rgba(0,0,0,0.05); border: 1px solid #eee; }
        .search-box input { border: none; outline: none; width: 100%; font-size: 14px; margin-left: 8px; font-family: 'Poppins', sans-serif; }

        /* TARJETAS PDV */
        #lista-pdv { padding: 0 10px; }
        .pdv-card {
            background: white; border-radius: 15px; padding: 14px; margin-bottom: 12px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.06); position: relative;
            overflow: hidden; border-left: 4px solid #ddd; transition: all 0.3s;
            animation: slideIn 0.3s ease-out;
        }
        @keyframes slideIn { from { opacity: 0; transform: translateY(20px); } to { opacity: 1; transform: translateY(0); } }
        .pdv-card[data-cat="A"] { border-left-color: #4361ee; }
        .pdv-card[data-cat="B"] { border-left-color: #f72585; }
        .pdv-card[data-cat="C"] { border-left-color: #7209b7; }
        .pdv-card[data-cat="D"] { border-left-color: #fca311; }
        .card-top { display: flex; justify-content: space-between; align-items: flex-start; margin-bottom: 8px; }
        .pdv-nombre { font-size: 15px; font-weight: 700; color: #333; flex: 1; padding-right: 8px; }
        .distancia-badge { background: #e0fbfc; color: #0077b6; padding: 3px 8px; border-radius: 8px; font-size: 11px; font-weight: 700; white-space: nowrap; }
        .pdv-info { display: flex; flex-wrap: wrap; gap: 6px; margin-bottom: 10px; font-size: 12px; color: #666; }
        .info-item { display: flex; align-items: center; gap: 4px; background: #f8f9fa; padding: 3px 6px; border-radius: 5px; }
        .info-item span { font-weight: 600; color: #444; }
        .pdv-detalles { display: grid; grid-template-columns: 1fr 1fr; gap: 6px; margin-bottom: 12px; padding-top: 8px; border-top: 1px solid #eee; }
        .detail-row { font-size: 11px; }
        .detail-label { color: #888; display: block; font-size: 9px; text-transform: uppercase; letter-spacing: 0.5px; }
        .detail-val { font-weight: 600; color: #333; }
        .detail-val.money { color: #2a9d8f; font-size: 13px; }
        .card-actions { display: flex; gap: 6px; }
        .btn-action { flex: 1; padding: 8px; border: none; border-radius: 8px; font-size: 12px; font-weight: 600; cursor: pointer; display: flex; align-items: center; justify-content: center; gap: 4px; color: white; }
        .btn-waze { background: var(--waze); color: #003366; }
        .btn-gmaps { background: var(--gmaps); }
        .btn-map { background: #6c757d; }

        .btn-cargar-mas { width: calc(100% - 20px); margin: 15px 10px; padding: 12px; background: white; border: 2px dashed var(--primary); color: var(--primary); border-radius: 10px; font-size: 14px; font-weight: 600; cursor: pointer; display: none; }
        .btn-volver { width: calc(100% - 20px); margin: 15px 10px; padding: 12px; background: var(--primary); color: white; border: none; border-radius: 10px; font-size: 14px; font-weight: 600; cursor: pointer; display: none; }

        /* BOTONES FLOTANTES */
        .floating-buttons {
            position: fixed;
            bottom: 20px;
            left: 50%;
            transform: translateX(-50%);
            display: flex;
            gap: 10px;
            z-index: 100;
            background: white;
            padding: 10px 15px;
            border-radius: 50px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.2);
        }
        .floating-buttons .btn-secondary { margin: 0; padding: 10px 16px; font-size: 13px; }

        /* MODALES */
        .modal-map { position: fixed; top: 0; left: 0; right: 0; bottom: 0; background: white; z-index: 2000; display: none; flex-direction: column; }
        .modal-map.active { display: flex; }
        .map-header { padding: 12px 15px; background: white; border-bottom: 1px solid #eee; display: flex; align-items: center; justify-content: space-between; gap: 10px; }
        .map-header-title { font-weight: 600; font-size: 15px; flex: 1; text-align: center; }
        .btn-close { background: #f1f3f5; border: none; width: 32px; height: 32px; border-radius: 50%; font-size: 18px; cursor: pointer; display: flex; align-items: center; justify-content: center; }
        #mini-map, #tracking-map, #ruta-map { flex: 1; width: 100%; }

        .loading { text-align: center; padding: 40px; display: none; }
        .spinner { width: 40px; height: 40px; border: 4px solid #f3f3f3; border-top: 4px solid var(--primary); border-radius: 50%; animation: spin 1s linear infinite; margin: 0 auto 15px; }
        @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
        .empty-state { text-align: center; padding: 60px 20px; color: #999; }
        .empty-state-icon { font-size: 48px; margin-bottom: 15px; opacity: 0.5; }
        .user-info { font-size: 11px; opacity: 0.9; margin-top: 3px; }
        @keyframes shake { 0%, 100% { transform: translateX(0); } 25% { transform: translateX(-10px); } 75% { transform: translateX(10px); } }

        /* PANEL DE SEGUIMIENTO */
        .tracking-panel { padding: 15px; background: white; border-radius: 12px; margin: 10px; box-shadow: 0 2px 10px rgba(0,0,0,0.05); }
        .tracking-info { display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px; }
        .tracking-distance { font-size: 28px; font-weight: 700; color: var(--primary); }
        .tracking-status { font-size: 12px; padding: 4px 10px; border-radius: 15px; font-weight: 600; }
        .status-cerca { background: #d4edda; color: #155724; }
        .status-lejos { background: #f8d7da; color: #721c24; }
        .tracking-update { font-size: 11px; color: #666; margin-top: 5px; text-align: right; }

        /* PANEL DE RUTA - CON COLAPSO */
        .ruta-panel {
            background: white;
            border-radius: 12px;
            margin: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.05);
            transition: all 0.3s;
        }
        .ruta-panel.collapsed { margin: 5px 10px; }
        .ruta-panel-header {
            padding: 12px 15px;
            background: #f8f9fa;
            border-radius: 12px 12px 0 0;
            display: flex;
            justify-content: space-between;
            align-items: center;
            cursor: pointer;
            border-bottom: 2px solid transparent;
            transition: all 0.3s;
        }
        .ruta-panel-header:hover { background: #e9ecef; }
        .ruta-panel.collapsed .ruta-panel-header { border-radius: 12px; border-bottom: none; }
        .ruta-panel-title { font-weight: 600; font-size: 14px; color: var(--text); }
        .ruta-panel-toggle {
            background: var(--primary);
            color: white;
            border: none;
            width: 28px;
            height: 28px;
            border-radius: 50%;
            font-size: 16px;
            cursor: pointer;
            display: flex;
            align-items: center;
            justify-content: center;
            transition: transform 0.3s;
        }
        .ruta-panel.collapsed .ruta-panel-toggle { transform: rotate(180deg); }
        .ruta-panel-content {
            padding: 15px;
            transition: all 0.3s;
        }
        .ruta-panel.collapsed .ruta-panel-content { display: none; }
        .ruta-stats { display: flex; gap: 8px; margin-bottom: 12px; }
        .ruta-stat { flex: 1; text-align: center; padding: 8px; background: #f8f9fa; border-radius: 8px; }
        .ruta-stat-val { font-size: 16px; font-weight: 700; color: var(--primary); }
        .ruta-stat-label { font-size: 10px; color: #666; }
        .ruta-lista { max-height: 180px; overflow-y: auto; }
        .ruta-item { display: flex; align-items: center; padding: 6px; border-bottom: 1px solid #eee; font-size: 12px; }
        .ruta-item:last-child { border-bottom: none; }
        .ruta-num { width: 22px; height: 22px; background: var(--primary); color: white; border-radius: 50%; display: flex; align-items: center; justify-content: center; margin-right: 8px; font-size: 11px; font-weight: bold; }
        .btn-ruta-accion { width: calc(100% - 20px); margin: 10px; padding: 12px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border: none; border-radius: 10px; font-size: 14px; font-weight: 600; cursor: pointer; display: none; }

        .map-legend { position: absolute; bottom: 15px; left: 50%; transform: translateX(-50%); background: white; padding: 6px 12px; border-radius: 15px; font-size: 11px; box-shadow: 0 2px 10px rgba(0,0,0,0.1); z-index: 1000; display: flex; gap: 12px; align-items: center; flex-wrap: wrap; justify-content: center; }
        .legend-item { display: flex; align-items: center; gap: 4px; }
        .legend-dot { width: 12px; height: 12px; border-radius: 50%; border: 2px solid white; box-shadow: 0 0 0 2px currentColor; }
        .legend-user { background: #06d6a0; color: #06d6a0; }
        .legend-pdv { background: #f72585; color: #f72585; }
        .legend-route { width: 20px; height: 3px; background: #667eea; }
    </style>
</head>
<body>
    <!-- LOGIN -->
    <div id="loginOverlay">
        <div class="login-box">
            <div style="font-size:48px;margin-bottom:10px;">🔐</div>
            <h2 class="login-title">PDV Finder Pro</h2>
            <p class="login-subtitle">Acceso restringido - Tecnico Comercial</p>
            <div class="login-error" id="loginError"> Usuario o contrasena incorrectos</div>
            <input type="text" class="login-input" id="loginUser" placeholder="Usuario" autocomplete="username">
            <input type="password" class="login-input" id="loginPass" placeholder="Contrasena" autocomplete="current-password" onkeypress="if(event.key==='Enter') validarLogin()">
            <button class="login-btn" onclick="validarLogin()"> Ingresar</button>
            <div class="login-footer"><p> 2026 - Uso exclusivo del equipo</p></div>
        </div>
    </div>

    <!-- APP CONTENT -->
    <div id="appContent">
        <div class="header">
            <button class="btn-logout" onclick="cerrarSesion()"> Salir</button>
            <h1> PDV Finder Pro</h1>
            <p class="user-info" id="userInfo">Tecnico Comercial</p>
        </div>

        <button class="btn-main" id="btnMain" onclick="cargarCercanos()">
            <span></span> Actualizar Ubicacion
        </button>

        <div class="contador" id="contador">
            Mostrando <strong id="mostrados">0</strong> de <strong id="total">0</strong> PDVs cercanos
        </div>

        <div class="modo-busqueda" id="modoBusqueda">
             Busqueda en TODA la base de datos
        </div>

        <div class="search-container">
            <div class="search-box">
                <span></span>
                <input type="text" id="buscador" placeholder="Buscar Nombre, Codigo o Recarga..." oninput="buscarEnTiempoReal()" disabled>
            </div>
        </div>

        <div id="loading" class="loading">
            <div class="spinner"></div>
            <p id="loadingText">Calculando distancias GPS...</p>
        </div>

        <div id="lista-pdv">
            <div class="empty-state">
                <div class="empty-state-icon"></div>
                <p>Toca "Actualizar Ubicacion"<br>para comenzar</p>
            </div>
        </div>

        <button class="btn-cargar-mas" id="btnCargarMas" onclick="cargarMas()">
             Cargar 15 PDVs mas
        </button>

        <button class="btn-volver" id="btnVolver" onclick="volverACercanos()">
             Volver a PDVs cercanos
        </button>

        <!-- BOTONES FLOTANTES -->
        <div class="floating-buttons">
            <button class="btn-secondary" onclick="abrirSeguimiento()">
                 Seguimiento
            </button>
            <button class="btn-secondary" onclick="abrirRutaLogica()">
                 Ver Ruta Logica
            </button>
        </div>

        <!-- MODAL SEGUIMIENTO -->
        <div class="modal-map" id="modalSeguimiento">
            <div class="map-header">
                <button class="btn-close" onclick="cerrarSeguimiento()">X</button>
                <div class="map-header-title"> Seguimiento en Vivo</div>
                <button class="btn-secondary" onclick="actualizarSeguimiento()" style="padding:6px 12px;font-size:12px;"> Actualizar</button>
            </div>
            <div id="tracking-map"></div>
            <div class="tracking-panel">
                <div class="tracking-info">
                    <div>
                        <div style="font-size:14px;font-weight:600;" id="trackingPdvNombre">Selecciona un PDV</div>
                        <div style="font-size:11px;color:#666;" id="trackingPdvInfo"></div>
                    </div>
                    <div class="tracking-distance" id="trackingDistancia">-- km</div>
                </div>
                <div style="display:flex;justify-content:space-between;align-items:center;">
                    <span class="tracking-status" id="trackingStatus">Esperando ubicacion...</span>
                    <div class="tracking-update" id="trackingUpdate"></div>
                </div>
            </div>
            <div class="map-legend">
                <div class="legend-item"><div class="legend-dot legend-user"></div> Tu ubicacion</div>
                <div class="legend-item"><div class="legend-dot legend-pdv"></div> PDV</div>
                <div class="legend-item"><div class="legend-route"></div> Distancia</div>
            </div>
        </div>

        <!-- MODAL RUTA LOGICA -->
        <div class="modal-map" id="modalRuta">
            <div class="map-header">
                <button class="btn-close" onclick="cerrarRuta()">X</button>
                <div class="map-header-title"> Ruta Logica - 15 PDVs</div>
                <button class="btn-secondary" onclick="recalcularRuta()" style="padding:6px 12px;font-size:12px;"> Recalcular</button>
            </div>
            <div id="ruta-map"></div>
            <div class="ruta-panel" id="rutaPanel">
                <div class="ruta-panel-header" onclick="toggleRutaPanel()">
                    <div class="ruta-panel-title"> Lista de PDVs en Ruta</div>
                    <button class="ruta-panel-toggle" id="rutaToggleBtn">▼</button>
                </div>
                <div class="ruta-panel-content">
                    <div class="ruta-stats">
                        <div class="ruta-stat"><div class="ruta-stat-val" id="rutaDistancia">--</div><div class="ruta-stat-label">Km Totales</div></div>
                        <div class="ruta-stat"><div class="ruta-stat-val" id="rutaTiempo">--</div><div class="ruta-stat-label">Tiempo Est.</div></div>
                        <div class="ruta-stat"><div class="ruta-stat-val" id="rutaPuntos">15</div><div class="ruta-stat-label">PDVs</div></div>
                    </div>
                    <div class="ruta-lista" id="rutaLista"></div>
                </div>
            </div>
            <button class="btn-ruta-accion" id="btnExportarRuta" onclick="exportarRuta()">
                 Exportar Primer Destino a Waze
            </button>
            <div class="map-legend">
                <div class="legend-item"><div class="legend-dot legend-user"></div> Inicio</div>
                <div class="legend-item"><div class="legend-dot legend-pdv"></div> PDV</div>
                <div class="legend-item"><div class="legend-route"></div> Ruta optima</div>
            </div>
        </div>

        <!-- MODAL MAPA SIMPLE -->
        <div class="modal-map" id="modalMap">
            <div class="map-header">
                <button class="btn-close" onclick="cerrarMapa()">X</button>
                <div class="map-header-title"> Ver ubicacion PDV</div>
            </div>
            <div id="mini-map"></div>
        </div>
    </div>

    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
    const USUARIOS = {
        "Jose Luis German": "SOjLg",
        "Rodrigo Medina": "AHRoRo",
        "Admin": "AdminOcc",
        "Invitado": "000000"
    };
    const todosLosPDVs = __PDV_DATA__;

    // CONFIGURACIÓN DE SESIÓN
    const SESSION_KEY = 'pdvFinderSesion';
    const USERNAME_KEY = 'pdvFinderUsuario';
    const DIAS_SESION = 7;
    const EXPIRACION = DIAS_SESION * 24 * 60 * 60 * 1000; // 7 días en ms

    let pdvsConDistancia = [];
    let pdvsMostrados = [];
    let resultadosBusqueda = [];
    let indiceCarga = 0;
    const CANTIDAD_POR_CARGA = 15;
    let userLat = null;
    let userLon = null;
    let modoBusquedaActivo = false;
    let timeoutBusqueda = null;
    let usuarioActual = null;

    // Variables para seguimiento
    let trackingWatchId = null;
    let trackingMap = null;
    let trackingUserMarker = null;
    let trackingPdvMarker = null;
    let trackingLine = null;
    let pdvSeleccionadoSeguimiento = null;

    // Variables para ruta
    let rutaMap = null;
    let rutaLayer = null;
    let rutaOptima = [];
    let rutaWatchId = null;
    let rutaUserMarker = null;

    // VERIFICAR SESIÓN AL CARGAR
    window.onload = function() {
        // 1. Recordar usuario permanentemente
        const usuarioGuardado = localStorage.getItem(USERNAME_KEY);
        if(usuarioGuardado) document.getElementById('loginUser').value = usuarioGuardado;

        // 2. Verificar sesión activa y no expirada
        const stored = localStorage.getItem(SESSION_KEY);
        if(stored) {
            try {
                const data = JSON.parse(stored);
                if(data.expira > Date.now()) {
                    usuarioActual = data.usuario;
                    iniciarApp();
                } else {
                    localStorage.removeItem(SESSION_KEY);
                }
            } catch(e) {
                localStorage.removeItem(SESSION_KEY);
            }
        }
    };

    function validarLogin() {
        const user = document.getElementById('loginUser').value.trim();
        const pass = document.getElementById('loginPass').value.trim();
        const error = document.getElementById('loginError');

        if(USUARIOS[user] === pass) {
            usuarioActual = user;

            // Guardar usuario permanentemente
            localStorage.setItem(USERNAME_KEY, user);

            // Guardar sesión con fecha de expiración
            localStorage.setItem(SESSION_KEY, JSON.stringify({
                usuario: user,
                token: pass,
                expira: Date.now() + EXPIRACION
            }));

            iniciarApp();
        } else {
            error.style.display = 'block';
            document.getElementById('loginPass').value = '';
            document.getElementById('loginPass').focus();
            const box = document.querySelector('.login-box');
            box.style.animation = 'shake 0.3s';
            setTimeout(() => box.style.animation = '', 300);
        }
    }

    function iniciarApp() {
        document.getElementById('loginOverlay').style.display = 'none';
        document.getElementById('appContent').style.display = 'block';
        document.getElementById('userInfo').textContent = usuarioActual;
        cargarCercanos();
    }

    function cerrarSesion() {
        if(confirm('Cerrar sesion? (Tu usuario seguira recordado)')) {
            localStorage.removeItem(SESSION_KEY);
            // NO borramos USERNAME_KEY para que siga recordando el usuario
            usuarioActual = null;
            document.getElementById('appContent').style.display = 'none';
            document.getElementById('loginOverlay').style.display = 'flex';
            document.getElementById('loginPass').value = '';
            document.getElementById('loginError').style.display = 'none';
            if(trackingWatchId) navigator.geolocation.clearWatch(trackingWatchId);
            if(rutaWatchId) navigator.geolocation.clearWatch(rutaWatchId);
        }
    }

    // FUNCIONES BASE
    function calcularDistancia(lat1, lon1, lat2, lon2) {
        const R = 6371;
        const dLat = (lat2 - lat1) * Math.PI / 180;
        const dLon = (lon2 - lon1) * Math.PI / 180;
        const a = Math.sin(dLat/2) * Math.sin(dLat/2) + Math.cos(lat1 * Math.PI / 180) * Math.cos(lat2 * Math.PI / 180) * Math.sin(dLon/2) * Math.sin(dLon/2);
        const c = 2 * Math.atan2(Math.sqrt(a), Math.sqrt(1-a));
        return R * c;
    }

    function cargarCercanos() {
        if (!navigator.geolocation) { alert("Tu navegador no soporta GPS"); return; }

        const loading = document.getElementById('loading');
        const btn = document.getElementById('btnMain');
        const buscador = document.getElementById('buscador');

        loading.style.display = 'block';
        btn.innerHTML = '<span></span> Ubicando...';
        buscador.disabled = true;
        modoBusquedaActivo = false;

        navigator.geolocation.getCurrentPosition(
            (pos) => {
                userLat = pos.coords.latitude;
                userLon = pos.coords.longitude;

                pdvsConDistancia = todosLosPDVs.map(p => ({ ...p, distancia: calcularDistancia(userLat, userLon, p.lat, p.lon) }));
                pdvsConDistancia.sort((a, b) => a.distancia - b.distancia);

                indiceCarga = 0;
                pdvsMostrados = [];
                cargarSiguienteLote();

                loading.style.display = 'none';
                btn.innerHTML = '<span></span> Actualizar Ubicacion';
                buscador.disabled = false;
                buscador.placeholder = "Buscar Nombre, Codigo o Recarga";
                document.getElementById('contador').style.display = 'block';
                actualizarContador();
                document.getElementById('btnCargarMas').style.display = pdvsConDistancia.length > CANTIDAD_POR_CARGA ? 'block' : 'none';
                document.getElementById('btnVolver').style.display = 'none';
            },
            (err) => {
                loading.style.display = 'none';
                btn.innerHTML = '<span></span> Actualizar Ubicacion';
                alert("Error GPS: " + err.message);
            },
            { enableHighAccuracy: true, timeout: 10000 }
        );
    }

    function cargarSiguienteLote() {
        const inicio = indiceCarga;
        const fin = Math.min(indiceCarga + CANTIDAD_POR_CARGA, modoBusquedaActivo ? resultadosBusqueda.length : pdvsConDistancia.length);
        const datosOrigen = modoBusquedaActivo ? resultadosBusqueda : pdvsConDistancia;
        for(let i = inicio; i < fin; i++) { pdvsMostrados.push(datosOrigen[i]); }
        indiceCarga = fin;
        renderizarLista(pdvsMostrados);
        actualizarContador();
        const total = modoBusquedaActivo ? resultadosBusqueda.length : pdvsConDistancia.length;
        document.getElementById('btnCargarMas').style.display = indiceCarga >= total ? 'none' : 'block';
    }

    function cargarMas() {
        const btn = document.getElementById('btnCargarMas');
        btn.innerHTML = ' Cargando...';
        btn.disabled = true;
        setTimeout(() => {
            cargarSiguienteLote();
            btn.innerHTML = ' Cargar 15 PDVs mas';
            btn.disabled = false;
            document.getElementById('lista-pdv').lastElementChild.scrollIntoView({ behavior: 'smooth', block: 'center' });
        }, 300);
    }

    function buscarEnTiempoReal() {
        clearTimeout(timeoutBusqueda);
        const texto = document.getElementById('buscador').value.trim();
        if(texto.length < 2) { if(texto.length === 0 && modoBusquedaActivo) volverACercanos(); return; }
        timeoutBusqueda = setTimeout(() => realizarBusquedaCompleta(texto), 300);
    }

    function realizarBusquedaCompleta(texto) {
        const loading = document.getElementById('loading');
        const textoLower = texto.toLowerCase();
        loading.style.display = 'block';
        document.getElementById('loadingText').textContent = 'Buscando en ' + todosLosPDVs.length + ' PDVs...';

        setTimeout(() => {
            resultadosBusqueda = todosLosPDVs.map(p => ({ ...p, distancia: userLat ? calcularDistancia(userLat, userLon, p.lat, p.lon) : 999999 }))
                .filter(p => p.nombre.toLowerCase().includes(textoLower) || p.codigo_pdv.includes(texto) || p.recarga.includes(texto) || p.ruta.toLowerCase().includes(textoLower));
            resultadosBusqueda.sort((a, b) => a.distancia - b.distancia);

            loading.style.display = 'none';
            modoBusquedaActivo = true;
            indiceCarga = 0;
            pdvsMostrados = [];
            document.getElementById('modoBusqueda').classList.add('active');
            document.getElementById('btnCargarMas').style.display = resultadosBusqueda.length > CANTIDAD_POR_CARGA ? 'block' : 'none';
            document.getElementById('btnVolver').style.display = resultadosBusqueda.length > 0 ? 'block' : 'none';

            if(resultadosBusqueda.length > 0) { cargarSiguienteLote(); }
            else {
                document.getElementById('lista-pdv').innerHTML = '<div class="empty-state"><div class="empty-state-icon">X</div><p>No se encontraron resultados</p></div>';
                document.getElementById('contador').style.display = 'none';
            }
        }, 100);
    }

    function volverACercanos() {
        document.getElementById('buscador').value = '';
        document.getElementById('modoBusqueda').classList.remove('active');
        modoBusquedaActivo = false;
        resultadosBusqueda = [];
        if(pdvsConDistancia.length > 0) {
            indiceCarga = 0; pdvsMostrados = []; cargarSiguienteLote();
            document.getElementById('btnCargarMas').style.display = pdvsConDistancia.length > CANTIDAD_POR_CARGA ? 'block' : 'none';
            document.getElementById('btnVolver').style.display = 'none';
            document.getElementById('contador').style.display = 'block';
        }
    }

    function actualizarContador() {
        const total = modoBusquedaActivo ? resultadosBusqueda.length : pdvsConDistancia.length;
        document.getElementById('mostrados').textContent = pdvsMostrados.length;
        document.getElementById('total').textContent = total;
    }

    function renderizarLista(datos) {
        const lista = document.getElementById('lista-pdv');
        if(datos.length === 0) return;
        if(indiceCarga <= CANTIDAD_POR_CARGA) lista.innerHTML = '';

        const nuevosDatos = datos.slice(lista.children.length);
        nuevosDatos.forEach(p => {
            let distStr = p.distancia < 999999 ? (p.distancia < 1 ? (p.distancia * 1000).toFixed(0) + ' m' : p.distancia.toFixed(2) + ' km') : 'N/A';
            const wazeUrl = 'https://waze.com/ul?ll=' + p.lat + ',' + p.lon + '&navigate=yes';
            const gmapsUrl = 'https://www.google.com/maps/dir/?api=1&destination=' + p.lat + ',' + p.lon;

            const card = document.createElement('div');
            card.className = 'pdv-card';
            card.setAttribute('data-cat', p.categoria);
            card.innerHTML = '<div class="card-top"><div class="pdv-nombre">' + p.nombre + '</div><div class="distancia-badge">' + distStr + '</div></div>' +
                '<div class="pdv-info"><div class="info-item"><span>' + p.codigo_pdv + '</span></div><div class="info-item"><span>' + p.recarga + '</span></div><div class="info-item"><span>' + p.dias + '</span></div></div>' +
                '<div class="pdv-detalles"><div class="detail-row"><span class="detail-label">Ruta</span><span class="detail-val">' + p.ruta + '</span></div>' +
                '<div class="detail-row"><span class="detail-label">Categoria</span><span class="detail-val">' + p.categoria + '</span></div>' +
                '<div class="detail-row" style="grid-column: span 2;"><span class="detail-label">Venta Promedio</span><span class="detail-val money">' + p.venta + '</span></div></div>' +
                '<div class="card-actions"><button class="btn-action btn-waze" onclick="window.open(\\'' + wazeUrl + '\\', \\'_blank\\')"> Waze</button>' +
                '<button class="btn-action btn-gmaps" onclick="window.open(\\'' + gmapsUrl + '\\', \\'_blank\\')"> Maps</button>' +
                '<button class="btn-action btn-map" onclick="verEnMapa(' + p.lat + ', ' + p.lon + ', \\'' + p.nombre + '\\')"> Ver</button></div>';
            lista.appendChild(card);
        });
    }

    function verEnMapa(lat, lon, nombre) {
        document.getElementById('modalMap').classList.add('active');
        setTimeout(() => {
            if(!window.miniMap) {
                window.miniMap = L.map('mini-map').setView([lat, lon], 17);
                L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png').addTo(window.miniMap);
            } else {
                window.miniMap.setView([lat, lon], 17);
            }
            L.marker([lat, lon]).addTo(window.miniMap).bindPopup(nombre).openPopup();
        }, 100);
    }

    function cerrarMapa() {
        document.getElementById('modalMap').classList.remove('active');
    }

    // SEGUIMIENTO EN VIVO
    function abrirSeguimiento() {
        if(!pdvSeleccionadoSeguimiento && pdvsConDistancia.length > 0) {
            const pdvMasCercano = pdvsConDistancia[0];
            pdvSeleccionadoSeguimiento = {
                lat: pdvMasCercano.lat,
                lon: pdvMasCercano.lon,
                nombre: pdvMasCercano.nombre,
                codigo: pdvMasCercano.codigo_pdv
            };
        }

        if(!pdvSeleccionadoSeguimiento) {
            alert('Primero actualiza tu ubicacion para ver PDVs cercanos');
            return;
        }

        document.getElementById('modalSeguimiento').classList.add('active');
        document.getElementById('trackingPdvNombre').textContent = pdvSeleccionadoSeguimiento.nombre;
        document.getElementById('trackingPdvInfo').textContent = 'Codigo: ' + pdvSeleccionadoSeguimiento.codigo;

        setTimeout(inicializarMapaSeguimiento, 100);
    }

    function inicializarMapaSeguimiento() {
        if(!trackingMap) {
            trackingMap = L.map('tracking-map').setView([userLat || 13.8, userLon || -89.2], 16);
            L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png').addTo(trackingMap);
        }

        if(trackingUserMarker) { trackingMap.removeLayer(trackingUserMarker); trackingUserMarker = null; }
        if(trackingPdvMarker) { trackingMap.removeLayer(trackingPdvMarker); trackingPdvMarker = null; }
        if(trackingLine) { trackingMap.removeLayer(trackingLine); trackingLine = null; }

        trackingPdvMarker = L.marker([pdvSeleccionadoSeguimiento.lat, pdvSeleccionadoSeguimiento.lon], {
            icon: L.divIcon({
                className: '',
                html: '<div style="background:#f72585;color:white;width:20px;height:20px;border-radius:50%;border:4px solid white;box-shadow:0 0 0 3px #f72585,0 3px 10px rgba(0,0,0,0.4);display:flex;align-items:center;justify-content:center;"><div style="width:8px;height:8px;background:white;border-radius:50%;"></div></div>',
                iconSize: [28, 28],
                iconAnchor: [14, 14]
            }),
            zIndexOffset: 1000
        }).addTo(trackingMap).bindPopup('<b>PDV: ' + pdvSeleccionadoSeguimiento.nombre + '</b>');

        iniciarTrackingGPS();
    }

    function iniciarTrackingGPS() {
        if(!navigator.geolocation) {
            document.getElementById('trackingStatus').textContent = 'GPS no disponible';
            document.getElementById('trackingStatus').className = 'tracking-status status-lejos';
            return;
        }

        if(trackingWatchId) navigator.geolocation.clearWatch(trackingWatchId);

        document.getElementById('trackingStatus').textContent = 'Obteniendo ubicacion...';
        document.getElementById('trackingStatus').className = 'tracking-status';

        trackingWatchId = navigator.geolocation.watchPosition(
            (pos) => {
                userLat = pos.coords.latitude;
                userLon = pos.coords.longitude;

                if(trackingUserMarker) {
                    trackingUserMarker.setLatLng([userLat, userLon]);
                } else {
                    trackingUserMarker = L.marker([userLat, userLon], {
                        icon: L.divIcon({
                            className: '',
                            html: '<div style="background:#06d6a0;color:white;width:20px;height:20px;border-radius:50%;border:4px solid white;box-shadow:0 0 0 3px #06d6a0,0 3px 10px rgba(0,0,0,0.4);display:flex;align-items:center;justify-content:center;"><div style="width:8px;height:8px;background:white;border-radius:50%;"></div></div>',
                            iconSize: [28, 28],
                            iconAnchor: [14, 14]
                        }),
                        zIndexOffset: 1001
                    }).addTo(trackingMap).bindPopup('<b>Tu ubicacion</b>');
                }

                const distancia = calcularDistancia(userLat, userLon, pdvSeleccionadoSeguimiento.lat, pdvSeleccionadoSeguimiento.lon);
                let distStr = distancia < 1 ? (distancia * 1000).toFixed(0) + ' m' : distancia.toFixed(2) + ' km';
                document.getElementById('trackingDistancia').textContent = distStr;

                const statusEl = document.getElementById('trackingStatus');
                if(distancia < 0.1) {
                    statusEl.textContent = ' MUY CERCA';
                    statusEl.className = 'tracking-status status-cerca';
                } else if(distancia < 0.5) {
                    statusEl.textContent = ' Cerca';
                    statusEl.className = 'tracking-status status-cerca';
                } else {
                    statusEl.textContent = ' Lejos';
                    statusEl.className = 'tracking-status status-lejos';
                }

                if(trackingLine) trackingMap.removeLayer(trackingLine);
                trackingLine = L.polyline([[userLat, userLon], [pdvSeleccionadoSeguimiento.lat, pdvSeleccionadoSeguimiento.lon]], {
                    color: '#667eea', weight: 3, opacity: 0.8, dashArray: '5, 5'
                }).addTo(trackingMap);

                const ahora = new Date();
                document.getElementById('trackingUpdate').textContent = 'Actualizado: ' + ahora.getHours().toString().padStart(2,'0') + ':' + ahora.getMinutes().toString().padStart(2,'0');

                const bounds = L.latLngBounds([[userLat, userLon], [pdvSeleccionadoSeguimiento.lat, pdvSeleccionadoSeguimiento.lon]]);
                trackingMap.fitBounds(bounds.pad(0.3));

            },
            (err) => {
                document.getElementById('trackingStatus').textContent = 'Error GPS';
                document.getElementById('trackingStatus').className = 'tracking-status status-lejos';
            },
            { enableHighAccuracy: true, timeout: 15000, maximumAge: 0 }
        );
    }

    function actualizarSeguimiento() {
        if(trackingWatchId) navigator.geolocation.clearWatch(trackingWatchId);
        if(trackingMap) { trackingMap.remove(); trackingMap = null; }
        trackingUserMarker = null;
        trackingPdvMarker = null;
        trackingLine = null;
        inicializarMapaSeguimiento();
    }

    function cerrarSeguimiento() {
        document.getElementById('modalSeguimiento').classList.remove('active');
        if(trackingWatchId) { navigator.geolocation.clearWatch(trackingWatchId); trackingWatchId = null; }
    }

    // RUTA LOGICA CON TRACKING
    function abrirRutaLogica() {
        if(!userLat || !userLon) {
            alert('Primero obtén tu ubicación tocando "Actualizar Ubicacion"');
            return;
        }
        document.getElementById('modalRuta').classList.add('active');
        document.getElementById('rutaPanel').classList.remove('collapsed');
        document.getElementById('rutaToggleBtn').textContent = '▼';
        calcularRutaLogica();
    }

    function toggleRutaPanel() {
        const panel = document.getElementById('rutaPanel');
        const btn = document.getElementById('rutaToggleBtn');
        panel.classList.toggle('collapsed');
        btn.textContent = panel.classList.contains('collapsed') ? '▲' : '▼';
        setTimeout(() => {
            if(rutaMap && rutaLayer) {
                rutaMap.invalidateSize();
                rutaMap.fitBounds(rutaLayer.getBounds().pad(0.15));
            }
        }, 350);
    }

    function calcularRutaLogica() {
        const loading = document.getElementById('loading');
        loading.style.display = 'block';
        document.getElementById('loadingText').textContent = 'Calculando ruta logica...';

        setTimeout(() => {
            const pdvsOrdenados = [...todosLosPDVs]
                .map(p => ({...p, distancia: calcularDistancia(userLat, userLon, p.lat, p.lon)}))
                .sort((a,b) => a.distancia - b.distancia)
                .slice(0, 15);

            rutaOptima = optimizarRutaVecinoCercano(userLat, userLon, pdvsOrdenados);
            dibujarRutaEnMapa();
            actualizarEstadisticasRuta();
            iniciarTrackingEnRuta();

            loading.style.display = 'none';
            document.getElementById('btnExportarRuta').style.display = rutaOptima.length > 0 ? 'block' : 'none';
        }, 500);
    }

    function optimizarRutaVecinoCercano(latInicio, lonInicio, pdvs) {
        if(pdvs.length === 0) return [];
        let pendientes = [...pdvs];
        let ruta = [];
        let latActual = latInicio, lonActual = lonInicio;

        while(pendientes.length > 0) {
            let mejorIdx = 0, mejorDist = Infinity;
            pendientes.forEach((p, idx) => {
                const d = Math.sqrt(Math.pow(p.lat - latActual, 2) + Math.pow(p.lon - lonActual, 2));
                if(d < mejorDist) { mejorDist = d; mejorIdx = idx; }
            });
            const siguiente = pendientes.splice(mejorIdx, 1)[0];
            ruta.push(siguiente);
            latActual = siguiente.lat;
            lonActual = siguiente.lon;
        }
        return ruta;
    }

    function dibujarRutaEnMapa() {
        if(!rutaMap) {
            rutaMap = L.map('ruta-map').setView([userLat, userLon], 14);
            L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png').addTo(rutaMap);
        } else {
            if(rutaLayer) rutaMap.removeLayer(rutaLayer);
            if(rutaUserMarker) rutaMap.removeLayer(rutaUserMarker);
            rutaMap.setView([userLat, userLon], 14);
        }

        L.marker([userLat, userLon], {
            icon: L.divIcon({
                className: '',
                html: '<div style="background:#06d6a0;color:white;width:24px;height:24px;border-radius:50%;border:4px solid white;box-shadow:0 0 0 3px #06d6a0,0 3px 10px rgba(0,0,0,0.4);display:flex;align-items:center;justify-content:center;font-weight:bold;font-size:11px;">INICIO</div>',
                iconSize: [32, 32],
                iconAnchor: [16, 16]
            }),
            zIndexOffset: 1000
        }).addTo(rutaMap).bindPopup('<b>Tu ubicacion actual</b>');

        const coords = [[userLat, userLon]];
        const listaHtml = [];

        rutaOptima.forEach((p, i) => {
            coords.push([p.lat, p.lon]);
            L.marker([p.lat, p.lon], {
                icon: L.divIcon({
                    className: '',
                    html: '<div style="background:#f72585;color:white;width:22px;height:22px;border-radius:50%;border:3px solid white;box-shadow:0 3px 10px rgba(0,0,0,0.4);display:flex;align-items:center;justify-content:center;font-weight:bold;font-size:12px;">' + (i+1) + '</div>',
                    iconSize: [28, 28],
                    iconAnchor: [14, 14]
                }),
                zIndexOffset: 1001
            }).addTo(rutaMap).bindPopup('<b>' + (i+1) + '. ' + p.nombre + '</b><br>Codigo: ' + p.codigo_pdv);

            const distStr = p.distancia < 1 ? (p.distancia*1000).toFixed(0)+' m' : p.distancia.toFixed(2)+' km';
            listaHtml.push('<div class="ruta-item"><div class="ruta-num">' + (i+1) + '</div><div><b>' + p.nombre + '</b><br><small style="color:#666">' + distStr + ' - ' + p.venta + '</small></div></div>');
        });

        rutaLayer = L.polyline(coords, { color: '#667eea', weight: 4, opacity: 0.8, lineCap: 'round' }).addTo(rutaMap);
        rutaMap.fitBounds(rutaLayer.getBounds().pad(0.15));
        document.getElementById('rutaLista').innerHTML = listaHtml.join('');
    }

    function iniciarTrackingEnRuta() {
        if(!navigator.geolocation) return;
        if(rutaWatchId) navigator.geolocation.clearWatch(rutaWatchId);

        rutaWatchId = navigator.geolocation.watchPosition(
            (pos) => {
                userLat = pos.coords.latitude;
                userLon = pos.coords.longitude;

                if(rutaUserMarker) {
                    rutaUserMarker.setLatLng([userLat, userLon]);
                } else {
                    rutaUserMarker = L.marker([userLat, userLon], {
                        icon: L.divIcon({
                            className: '',
                            html: '<div style="background:#06d6a0;color:white;width:20px;height:20px;border-radius:50%;border:4px solid white;box-shadow:0 0 0 3px #06d6a0,0 3px 10px rgba(0,0,0,0.4);display:flex;align-items:center;justify-content:center;"><div style="width:8px;height:8px;background:white;border-radius:50%;"></div></div>',
                            iconSize: [28, 28],
                            iconAnchor: [14, 14]
                        }),
                        zIndexOffset: 1002
                    }).addTo(rutaMap).bindPopup('<b>Tu ubicacion</b>');
                }
                actualizarEstadisticasRuta();
            },
            (err) => console.error('Error tracking ruta:', err),
            { enableHighAccuracy: true, timeout: 15000, maximumAge: 0 }
        );
    }

    function actualizarEstadisticasRuta() {
        if(rutaOptima.length === 0) return;
        let distTotal = 0;
        let latAnt = userLat, lonAnt = userLon;
        rutaOptima.forEach(p => {
            distTotal += calcularDistancia(latAnt, lonAnt, p.lat, p.lon);
            latAnt = p.lat; lonAnt = p.lon;
        });

        document.getElementById('rutaDistancia').textContent = distTotal.toFixed(1) + ' km';
        document.getElementById('rutaPuntos').textContent = rutaOptima.length;

        const tiempoMin = Math.ceil((distTotal/30)*60 + rutaOptima.length*10);
        const horas = Math.floor(tiempoMin / 60);
        const mins = tiempoMin % 60;
        document.getElementById('rutaTiempo').textContent = horas > 0 ? horas + 'h ' + mins + 'm' : mins + ' min';
    }

    function recalcularRuta() {
        if(!navigator.geolocation) { alert('GPS no disponible'); return; }
        const loading = document.getElementById('loading');
        loading.style.display = 'block';
        document.getElementById('loadingText').textContent = 'Obteniendo nueva ubicacion...';

        navigator.geolocation.getCurrentPosition(
            (pos) => {
                userLat = pos.coords.latitude;
                userLon = pos.coords.longitude;
                calcularRutaLogica();
            },
            (err) => {
                loading.style.display = 'none';
                alert('Error: ' + err.message);
            },
            { enableHighAccuracy: true, timeout: 10000 }
        );
    }

    function exportarRuta() {
        if(rutaOptima.length === 0) return;
        const primerPdv = rutaOptima[0];
        const wazeUrl = 'https://waze.com/ul?ll=' + primerPdv.lat + ',' + primerPdv.lon + '&navigate=yes';
        window.open(wazeUrl, '_blank');
    }

    function cerrarRuta() {
        document.getElementById('modalRuta').classList.remove('active');
        if(rutaWatchId) { navigator.geolocation.clearWatch(rutaWatchId); rutaWatchId = null; }
    }
</script>
</body>
</html>
'''

# Reemplazar datos
html_content = html_content.replace("__PDV_DATA__", json.dumps(pdv_data))

# ────────────────────────────────────────────
# 4. EXPORTAR
# ────────────────────────────────────────────
carpeta = "pdv_finder_pro_final"
os.makedirs(carpeta, exist_ok=True)

with open(os.path.join(carpeta, "index.html"), 'w', encoding='utf-8') as f:
    f.write(html_content)

nombre_zip = f"PDV_Finder_Pro_Final_{datetime.now().strftime('%Y%m%d_%H%M')}.zip"
with zipfile.ZipFile(nombre_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(os.path.join(carpeta, "index.html"), "index.html")

files.download(nombre_zip)
print(" PDV Finder PRO - VERSION FINAL listo")
print(f" Total PDVs: {total_pdv}")
print("\n MEJORAS IMPLEMENTADAS:")
print("   [1] Seguimiento en vivo corregido - Marcadores grandes y visibles")
print("   [2] Solo boton flotante de Seguimiento (eliminado el fijo)")
print("   [3] Ruta Logica con tracking en tiempo real de tu ubicacion")
print("   [4] Panel de Ruta minimizable/colapsable para ver mas el mapa")
print("   [5] Marcadores mas grandes con mejor contraste")
print("\n Usuarios:")
print("   - Jose Luis German / SOjLg")
print("   - Rodrigo Medina / AHRoRo")
print("   - Admin / AdminOcc")
print("   - Invitado / 000000")

 PDV FINDER PRO - VERSION FINAL

 Selecciona tu archivo Excel:


Saving Rutas y frecuencias CM ALL COMO.xlsx to Rutas y frecuencias CM ALL COMO.xlsx

 6560 PDV cargados correctamente


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 PDV Finder PRO - VERSION FINAL listo
 Total PDVs: 6560

 MEJORAS IMPLEMENTADAS:
   [1] Seguimiento en vivo corregido - Marcadores grandes y visibles
   [2] Solo boton flotante de Seguimiento (eliminado el fijo)
   [3] Ruta Logica con tracking en tiempo real de tu ubicacion
   [4] Panel de Ruta minimizable/colapsable para ver mas el mapa
   [5] Marcadores mas grandes con mejor contraste

 Usuarios:
   - Jose Luis German / SOjLg
   - Rodrigo Medina / AHRoRo
   - Admin / AdminOcc
   - Invitado / 000000
